第一部分：日维度流量与购买行为分析

分析问题

1. 每天有多少浏览行为和活跃用户？
2. 每位活跃用户平均产生多少次浏览？
3. 哪天浏览量增长明显？
4. 浏览量增长时，活跃人数和浏览深度如何变化？
5. 每天有多少购买行为和购买用户？

指标定义
PV：浏览行为记录数。
活跃用户：当天发生任意有效行为的去重用户数。
浏览用户：当天发生浏览行为的去重用户数。
购买行为次数：当天 buy 类型的记录数，不直接等同于订单数。
购买用户：当天发生购买行为的去重用户数。
每活跃用户浏览次数：当天 PV / 当天活跃用户数。

In [ ]:
from pathlib import Path

import duckdb
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
db_file = Path("../data/database/taobao.duckdb").resolve()

print("当前工作目录：", Path.cwd())
print("准备读取的数据库：", db_file)

if not db_file.is_file():
    raise FileNotFoundError(
        f"数据库文件不存在：{db_file}\n"
        "请确认 Notebook 的工作目录是项目的 notebooks/，"
        "并确认清洗后的数据库已保存在 data/database/。"
    )

# 如果是在复用已打开过该数据库的 Notebook，先关闭旧连接。
if "con" in globals():
    con.close()

con = duckdb.connect(str(db_file), read_only=True)

print("\n表结构：")
display(con.execute("DESCRIBE user_behavior").df())

print("\n前 5 行样例：")
display(
    con.execute(
        "SELECT * FROM user_behavior LIMIT 5"
    ).df()
)

In [ ]:
overview = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT user_id) AS total_users,
        MIN(behavior_date) AS first_date,
        MAX(behavior_date) AS last_date,
        COUNT(DISTINCT behavior_date) AS observed_days,

        COUNT(*) FILTER (WHERE behavior_type = 'pv') AS pv_records,
        COUNT(*) FILTER (WHERE behavior_type = 'cart') AS cart_records,
        COUNT(*) FILTER (WHERE behavior_type = 'fav') AS fav_records,
        COUNT(*) FILTER (WHERE behavior_type = 'buy') AS buy_records

    FROM user_behavior
""").df()

display(overview)

In [ ]:
behavior_summary = con.execute("""
    SELECT
        behavior_type,
        COUNT(*) AS behavior_records,
        COUNT(DISTINCT user_id) AS behavior_users
    FROM user_behavior
    GROUP BY behavior_type
    ORDER BY CASE behavior_type
        WHEN 'pv' THEN 1
        WHEN 'cart' THEN 2
        WHEN 'fav' THEN 3
        WHEN 'buy' THEN 4
    END
""").df()

total_records = int(overview.loc[0, "total_records"])
total_users = int(overview.loc[0, "total_users"])

behavior_summary["record_share_pct"] = (
    behavior_summary["behavior_records"] / total_records * 100
).round(2)

behavior_summary["user_share_pct"] = (
    behavior_summary["behavior_users"] / total_users * 100
).round(2)

display(behavior_summary)

print(
    "四类行为记录数合计：",
    f"{behavior_summary['behavior_records'].sum():,}"
)
print(
    "购买用户数：",
    f"{behavior_summary.loc[behavior_summary['behavior_type'] == 'buy', 'behavior_users'].iloc[0]:,}"
)

In [ ]:
daily_activity = con.execute("""
    SELECT
        behavior_date,

        COUNT(*) AS total_records,

        COUNT(*) FILTER (
            WHERE behavior_type = 'pv'
        ) AS pv_records,

        COUNT(DISTINCT user_id) AS active_users,

        COUNT(DISTINCT user_id) FILTER (
            WHERE behavior_type = 'pv'
        ) AS pv_users,

        COUNT(*) FILTER (
            WHERE behavior_type = 'buy'
        ) AS buy_records,

        COUNT(DISTINCT user_id) FILTER (
            WHERE behavior_type = 'buy'
        ) AS buy_users

    FROM user_behavior
    GROUP BY behavior_date
    ORDER BY behavior_date
""").df()

daily_activity["pv_per_active_user"] = (
    daily_activity["pv_records"]
    / daily_activity["active_users"]
).round(2)

display(daily_activity)

In [ ]:
print("汇总天数：", len(daily_activity))
print(
    "汇总日期范围：",
    daily_activity["behavior_date"].min(),
    "至",
    daily_activity["behavior_date"].max()
)

print(
    "日 PV 合计：",
    f"{daily_activity['pv_records'].sum():,}"
)

print(
    "全表 PV：",
    f"{int(overview.loc[0, 'total_records']):,}"
)

In [ ]:
processed_dir = Path("../data/processed").resolve()
processed_dir.mkdir(parents=True, exist_ok=True)

daily_activity_path = processed_dir / "daily_activity.csv"

daily_activity.to_csv(
    daily_activity_path,
    index=False,
    encoding="utf-8-sig"
)

print("日级汇总表已保存：", daily_activity_path)
print("文件大小：", daily_activity_path.stat().st_size, "bytes")

In [ ]:
# ==========================
# 1. 绘图基础设置
# ==========================

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURE_DIR
from src.plotting import setup_chinese_font
setup_chinese_font()

# ==========================
# 2. 检查并准备绘图数据
# ==========================

if "daily_activity" not in globals():
    raise NameError(
        "当前 Notebook 中没有找到 daily_activity，"
        "请先运行日级汇总代码。"
    )

plot_data = daily_activity.copy()

plot_data["behavior_date"] = pd.to_datetime(
    plot_data["behavior_date"]
)

plot_data = (
    plot_data
    .sort_values("behavior_date")
    .reset_index(drop=True)
)


required_columns = [
    "behavior_date",
    "pv_records",
    "active_users",
    "pv_per_active_user",
]

missing_columns = [
    column
    for column in required_columns
    if column not in plot_data.columns
]

if missing_columns:
    raise KeyError(
        f"缺少绘图所需字段：{missing_columns}"
    )


# ==========================
# 3. 定义颜色
# ==========================

COLOR_PV = "#2878B5"          # 蓝色：日 PV
COLOR_ACTIVE = "#3A9D5D"      # 绿色：日活跃用户
COLOR_AVG_PV = "#D88924"      # 橙色：人均 PV
COLOR_GRID = "#D9DEE5"        # 浅灰蓝：网格线
COLOR_TEXT = "#263238"        # 深灰：标题和文字


# ==========================
# 4. 创建三个子图
# ==========================

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(12, 10),
    sharex=True,
)

fig.patch.set_facecolor("white")


# ==========================
# 5. 绘制日 PV
# ==========================

axes[0].plot(
    plot_data["behavior_date"],
    plot_data["pv_records"],
    color=COLOR_PV,
    marker="o",
    markersize=5,
    linewidth=2.4,
    label="日 PV",
)

axes[0].set_title(
    "日 PV 趋势",
    loc="left",
    color=COLOR_TEXT,
    fontweight="bold",
    pad=10,
)

axes[0].set_ylabel("PV 记录数")

axes[0].grid(
    axis="y",
    color=COLOR_GRID,
    linestyle="--",
    linewidth=0.8,
    alpha=0.8,
)

axes[0].legend(
    loc="upper left",
    frameon=False,
)


# ==========================
# 6. 绘制日活跃用户
# ==========================

axes[1].plot(
    plot_data["behavior_date"],
    plot_data["active_users"],
    color=COLOR_ACTIVE,
    marker="o",
    markersize=5,
    linewidth=2.4,
    label="日活跃用户",
)

axes[1].set_title(
    "日活跃用户趋势",
    loc="left",
    color=COLOR_TEXT,
    fontweight="bold",
    pad=10,
)

axes[1].set_ylabel("去重用户数")

axes[1].grid(
    axis="y",
    color=COLOR_GRID,
    linestyle="--",
    linewidth=0.8,
    alpha=0.8,
)

axes[1].legend(
    loc="upper left",
    frameon=False,
)


# ==========================
# 7. 绘制人均 PV
# ==========================

axes[2].plot(
    plot_data["behavior_date"],
    plot_data["pv_per_active_user"],
    color=COLOR_AVG_PV,
    marker="o",
    markersize=5,
    linewidth=2.4,
    label="人均 PV",
)

axes[2].set_title(
    "人均 PV 趋势",
    loc="left",
    color=COLOR_TEXT,
    fontweight="bold",
    pad=10,
)

axes[2].set_ylabel("PV / 活跃用户")
axes[2].set_xlabel("日期")

axes[2].grid(
    axis="y",
    color=COLOR_GRID,
    linestyle="--",
    linewidth=0.8,
    alpha=0.8,
)

axes[2].legend(
    loc="upper left",
    frameon=False,
)


# ==========================
# 8. 统一优化坐标轴外观
# ==========================

for ax in axes:
    # 去掉右侧和顶部边框，减少视觉干扰
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # 左侧和底部边框使用浅灰色
    ax.spines["left"].set_color("#B8C2CC")
    ax.spines["bottom"].set_color("#B8C2CC")

    # 调整刻度颜色
    ax.tick_params(
        axis="both",
        colors="#455A64",
    )


# ==========================
# 9. 优化日期横轴
# ==========================

# 当前只有 9 天数据，因此每天显示一个标签
axes[2].xaxis.set_major_locator(
    mdates.DayLocator(interval=1)
)

# 使用中文日期格式，例如：11月25日
axes[2].xaxis.set_major_formatter(
    mdates.DateFormatter("%m月%d日")
)

# 日期标签旋转 0 度，横向显示
plt.setp(
    axes[2].get_xticklabels(),
    rotation=0,
    ha="center",
)


# ==========================
# 10. 设置总标题和副标题
# ==========================

fig.suptitle(
    "淘宝用户行为日趋势",
    fontsize=17,
    fontweight="bold",
    color=COLOR_TEXT,
    y=0.98,
)

fig.text(
    0.5,
    0.945,
    "PV、活跃用户与人均 PV（2017年11月25日—12月3日）",
    ha="center",
    fontsize=11,
    color="#607D8B",
)


# ==========================
# 11. 调整布局
# ==========================

fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.09,
    hspace=0.38,
)


# ==========================
# 12. 保存并显示图表
# ==========================

figure_dir = Path("../outputs/figures").resolve()
figure_dir.mkdir(
    parents=True,
    exist_ok=True,
)

figure_path = figure_dir / "daily_trend.png"

fig.savefig(
    figure_path,
    dpi=180,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

plt.close(fig)

print("优化后的趋势图已保存：", figure_path)
print("文件是否存在：", figure_path.is_file())

In [ ]:
# 1. 检查数据库连接是否存在
if "con" not in globals():
    raise NameError(
        "当前 Notebook 中没有找到 DuckDB 连接 con，"
        "请先运行数据库连接代码。"
    )


# 2. 按北京时间小时汇总 9 天行为数据
hourly_activity = con.execute("""
    SELECT
        behavior_hour,

        COUNT(*) AS total_records,

        COUNT(*) FILTER (
            WHERE behavior_type = 'pv'
        ) AS pv_records,

        COUNT(DISTINCT user_id) AS active_users,

        COUNT(DISTINCT user_id) FILTER (
            WHERE behavior_type = 'pv'
        ) AS pv_users,

        COUNT(*) FILTER (
            WHERE behavior_type = 'buy'
        ) AS buy_records,

        COUNT(DISTINCT user_id) FILTER (
            WHERE behavior_type = 'buy'
        ) AS buy_users

    FROM user_behavior

    GROUP BY behavior_hour
    ORDER BY behavior_hour
""").df()


# 3. 计算小时人均 PV
hourly_activity["pv_per_active_user"] = (
    hourly_activity["pv_records"]
    / hourly_activity["active_users"]
).round(2)


# 4. 显示小时级汇总表
display(hourly_activity.head())

In [ ]:
print("小时数：", len(hourly_activity))
print(
    "小时范围：",
    hourly_activity["behavior_hour"].min(),
    "至",
    hourly_activity["behavior_hour"].max()
)

print(
    "小时 PV 合计：",
    f"{hourly_activity['pv_records'].sum():,}"
)

print(
    "日级 PV 合计：",
    f"{daily_activity['pv_records'].sum():,}"
)



In [ ]:
hourly_peak_summary = {}

for metric in [
    "pv_records",
    "active_users",
    "pv_per_active_user",
    "buy_records",
    "buy_users",
]:
    min_idx = hourly_activity[metric].idxmin()
    max_idx = hourly_activity[metric].idxmax()

    hourly_peak_summary[metric] = {
        "min_hour": int(
            hourly_activity.loc[min_idx, "behavior_hour"]
        ),
        "min_value": hourly_activity.loc[min_idx, metric],
        "max_hour": int(
            hourly_activity.loc[max_idx, "behavior_hour"]
        ),
        "max_value": hourly_activity.loc[max_idx, metric],
    }

hourly_peak_summary

In [ ]:
processed_dir = Path("../data/processed").resolve()
processed_dir.mkdir(
    parents=True,
    exist_ok=True,
)

hourly_activity_path = processed_dir / "hourly_activity.csv"

hourly_activity.to_csv(
    hourly_activity_path,
    index=False,
    encoding="utf-8-sig",
)

print("小时级汇总表已保存：", hourly_activity_path)
print(
    "文件是否存在：",
    hourly_activity_path.is_file()
)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. 复制并按小时排序，不修改原始汇总表
plot_data = (
    hourly_activity
    .sort_values("behavior_hour")
    .reset_index(drop=True)
    .copy()
)

# 2. 确认包含完整的 0～23 时
if plot_data["behavior_hour"].tolist() != list(range(24)):
    raise ValueError("小时数据不是完整的 0～23 时，请先检查 hourly_activity。")

# 3. 中文字体设置；若中文显示为方框，之后再按本机字体调整
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

# 4. 每个指标独立纵轴，不使用双轴
series = [
    ("pv_records", "PV 记录数", "#2878B5"),
    ("active_users", "活跃用户数（跨 9 天去重）", "#3A9D5D"),
    ("buy_records", "购买行为记录数", "#D88924"),
]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for ax, (column, title, color) in zip(axes, series):
    ax.plot(
        plot_data["behavior_hour"],
        plot_data[column],
        color=color,
        marker="o",
        markersize=4,
        linewidth=2,
    )
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_ylabel("数量")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda value, _: f"{value / 10_000:.0f}万")
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# 每隔两小时显示一个标签，21 时峰值仍可从数据点观察
axes[-1].set_xticks(range(0, 24, 2))
axes[-1].set_xlim(0, 23)
axes[-1].set_xlabel("北京时间（小时；21 表示 21:00～21:59）")

fig.suptitle(
    "淘宝用户行为：24 小时分布",
    fontsize=16,
    fontweight="bold",
)
fig.text(
    0.5, 0.94,
    "2017-11-25 至 2017-12-03｜相同钟点的 9 天数据合并统计",
    ha="center",
    fontsize=10,
    color="#607D8B",
)
fig.subplots_adjust(top=0.89, bottom=0.09, hspace=0.40)

figure_path = Path("../outputs/figures/hourly_trend.png").resolve()
figure_path.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

print("图表已保存：", figure_path)
print("文件是否存在：", figure_path.is_file())

In [ ]:
con.close()
print("只读连接已关闭。")